# MLflow ML Trading Experiment Review

This notebook reviews completed Dagster-managed ML trading experiments. It does not build features, train models, score, or run backtests. Dagster owns execution and MLflow owns experiment tracking plus artifact storage.

In [ ]:
from __future__ import annotations

import mlflow
import pandas as pd

from quant_orchestrator.research_tools import load_latest_mlflow_experiment_artifacts
from quant_orchestrator.tracking import DEFAULT_TRACKING_URI

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

MLFLOW_EXPERIMENT = 'ml_trading'
BASELINE_EXPERIMENT_NAME = 'gpu_rf_shared_book_1t_dagster_smoke'
AE_EXPERIMENT_NAME = 'gpu_rf_autoencoder_shared_book_1t_dagster_smoke'

mlflow.set_tracking_uri(DEFAULT_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
if experiment is None:
    raise RuntimeError(f'MLflow experiment not found: {MLFLOW_EXPERIMENT}')
print({'tracking_uri': DEFAULT_TRACKING_URI, 'experiment_id': experiment.experiment_id, 'experiment_name': experiment.name})

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['start_time DESC'],
    max_results=100,
)
run_cols = [
    'tags.quant_orchestrator.experiment_name',
    'tags.quant_orchestrator.mode',
    'tags.quant_orchestrator.provider',
    'metrics.best_sharpe',
    'metrics.best_total_return',
    'metrics.best_max_drawdown',
    'metrics.trained_models',
    'metrics.strategy_sources',
    'metrics.elapsed_seconds',
    'status',
    'start_time',
    'run_id',
]
run_cols = [col for col in run_cols if col in runs.columns]
mlflow_runs = runs[run_cols].copy()
display(mlflow_runs)

In [ ]:
baseline = load_latest_mlflow_experiment_artifacts(BASELINE_EXPERIMENT_NAME, mlflow_experiment=MLFLOW_EXPERIMENT)
ae = load_latest_mlflow_experiment_artifacts(AE_EXPERIMENT_NAME, mlflow_experiment=MLFLOW_EXPERIMENT)

baseline_summary = baseline['backtest_summary'].assign(experiment_name=BASELINE_EXPERIMENT_NAME)
ae_summary = ae['backtest_summary'].assign(experiment_name=AE_EXPERIMENT_NAME)
comparison = pd.concat([baseline_summary, ae_summary], ignore_index=True)

cols = [
    'experiment_name', 'strategy_source', 'source', 'family', 'variant', 'top_k',
    'total_return', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure',
    'trades', 'signal_events',
]
cols = [col for col in cols if col in comparison.columns]
display(comparison[cols].sort_values(['sharpe', 'total_return'], ascending=False).head(25))

In [ ]:
best_by_experiment = (
    comparison[cols]
    .sort_values(['experiment_name', 'sharpe', 'total_return'], ascending=[True, False, False])
    .groupby('experiment_name', as_index=False)
    .head(1)
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(best_by_experiment)

if set(best_by_experiment['experiment_name']) >= {BASELINE_EXPERIMENT_NAME, AE_EXPERIMENT_NAME}:
    base = best_by_experiment.loc[best_by_experiment['experiment_name'].eq(BASELINE_EXPERIMENT_NAME)].iloc[0]
    ae_best = best_by_experiment.loc[best_by_experiment['experiment_name'].eq(AE_EXPERIMENT_NAME)].iloc[0]
    deltas = {
        'ae_minus_classifier_sharpe': float(ae_best['sharpe']) - float(base['sharpe']),
        'ae_minus_classifier_total_return': float(ae_best['total_return']) - float(base['total_return']),
        'ae_minus_classifier_max_drawdown': float(ae_best['max_drawdown']) - float(base['max_drawdown']),
    }
    print(deltas)

In [ ]:
from IPython.display import Markdown, display

analysis_lines = [
    '## Written Analysis',
    '',
    f'- MLflow experiment: `{MLFLOW_EXPERIMENT}`.',
    f'- Baseline artifact: `{BASELINE_EXPERIMENT_NAME}`.',
    f'- AE artifact: `{AE_EXPERIMENT_NAME}`.',
    f'- MLflow runs loaded: {len(mlflow_runs)}.',
]
if not best_by_experiment.empty:
    for row in best_by_experiment.itertuples(index=False):
        analysis_lines.append(
            f'- Best `{row.experiment_name}` row: {row.strategy_source} / {row.variant} top_k={int(row.top_k)}; '
            f'total_return={row.total_return:.2%}, sharpe={row.sharpe:.2f}, max_drawdown={row.max_drawdown:.2%}.'
        )
analysis_lines.extend([
    '',
    'Interpretation:',
    '- This notebook is intentionally read-only with respect to training and backtesting.',
    '- Use Dagster for running experiments and MLflow for comparing tracked runs.',
    '- Detailed tables are downloaded from the MLflow run artifacts.',
])
analysis_markdown = '\n'.join(analysis_lines)
display(Markdown(analysis_markdown))